# FitTrack: VLM Fine-tuning for Image -> Calorie Estimation

End-to-end pipeline for the work described in
[`docs/spec-vlm-calorie-estimation.md`](../docs/spec-vlm-calorie-estimation.md):

1. Download Nutrition5k (RGB images + metadata only, no depth/video for v1)
2. Parse dish metadata
3. Train/test split by dish ID
4. Build LLaVA-style instruction-tuning JSONL
5. Load `Qwen2-VL-2B-Instruct` in 4-bit (QLoRA)
6. Apply LoRA adapters
7. Train
8. Save the adapter
9. Evaluate on the held-out test split (MAE / MAPE on calories)

**Run this on Colab with a GPU runtime** (Runtime -> Change runtime type -> GPU).
Free T4 should work for the 2B model in 4-bit; bump to Colab Pro / A100 if
you hit OOM errors.

A few cells below have a `# VERIFY:` comment on assumptions (exact
Nutrition5k file layout, official split file names) that should be checked
against the actual downloaded data on first run -- the dataset repo's
structure is reproduced from its README/paper, not re-verified against a
live download as part of writing this notebook.

## 1. Setup

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes trl \
    qwen-vl-utils datasets pillow huggingface_hub pandas

In [ ]:
import os
import json
import random
import csv
import re
from pathlib import Path
from getpass import getpass

import torch
import pandas as pd
from PIL import Image

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU detected -- training will be impractically slow. "
          "Switch the Colab runtime to GPU.")

# Needed if you plan to push the trained adapter to the Hub at the end.
from huggingface_hub import login
hf_token = getpass("Hugging Face token (leave blank to skip login): ")
if hf_token:
    login(hf_token)

## 2. Download Nutrition5k (RGB images + metadata only)

Per the spec, v1 uses only the top-down RGB image per dish -- not the
depth maps or side-angle videos, which are much larger and unused here.

`gsutil` ships preinstalled on Colab. The bucket is public, so no GCP auth
is required to read it.

`# VERIFY:` the exact path under `imagery/` for the overhead RGB images --
this is `imagery/realsense_overhead/<dish_id>/rgb.png` per the dataset
README at the time this spec was written; confirm against
https://github.com/google-research-datasets/Nutrition5k before relying on
it, in case the bucket layout has changed.

In [ ]:
DATA_DIR = Path("./data/nutrition5k")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Metadata CSVs (small, has per-dish calories/macros/ingredients).
!gsutil -m cp -r gs://nutrition5k_dataset/nutrition5k_dataset/metadata {DATA_DIR}/

# Overhead RGB images only (skip side-angle videos and depth -- much larger,
# unused in this v1 pipeline).
!gsutil -m cp -r gs://nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead {DATA_DIR}/imagery_overhead

print("Downloaded contents:")
!find {DATA_DIR} -maxdepth 2 | head -20

## 3. Parse dish metadata

Nutrition5k's metadata CSVs are *not* fixed-width -- each row is
`dish_id, total_calories, total_mass, total_fat, total_carb, total_protein,
num_ingrs, (ingr_name, ingr_grams, ingr_calories, ingr_fat, ingr_carb,
ingr_protein) * num_ingrs`, so a plain `pandas.read_csv` will misalign
columns. Parsing it manually below.

`# VERIFY:` the exact per-ingredient field order/count against the dataset
README -- reproduced from the paper's description, not re-checked against a
live file in this notebook.

In [ ]:
def parse_metadata_csv(path):
    rows = []
    with open(path, newline="") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 6:
                continue
            dish_id = row[0]
            try:
                total_calories = float(row[1])
                total_mass = float(row[2])
                total_fat = float(row[3])
                total_carb = float(row[4])
                total_protein = float(row[5])
            except ValueError:
                continue  # header row or malformed line

            # Remaining fields are repeating ingredient tuples. The exact
            # tuple width (6 vs 7 fields per ingredient) varies by dataset
            # version -- detect it from the remainder length.
            remainder = row[6:]
            ingredient_names = []
            # Most Nutrition5k releases use 7 fields per ingredient:
            # (ingr_id, ingr_name, ingr_grams, ingr_calories, ingr_fat,
            #  ingr_carb, ingr_protein)
            step = 7
            for i in range(0, len(remainder) - step + 1, step):
                name = remainder[i + 1].strip()
                if name:
                    ingredient_names.append(name)

            rows.append({
                "dish_id": dish_id,
                "calories": total_calories,
                "mass_g": total_mass,
                "fat_g": total_fat,
                "carb_g": total_carb,
                "protein_g": total_protein,
                "ingredients": ingredient_names,
            })
    return rows

metadata_files = list((DATA_DIR / "metadata").glob("dish_metadata_*.csv"))
print("Found metadata files:", metadata_files)

all_rows = []
for f in metadata_files:
    all_rows.extend(parse_metadata_csv(f))

df = pd.DataFrame(all_rows)
print(f"Parsed {len(df)} dishes")
df.head()

In [ ]:
def image_path_for(dish_id):
    return DATA_DIR / "imagery_overhead" / dish_id / "rgb.png"

df["image_path"] = df["dish_id"].apply(lambda d: str(image_path_for(d)))
df["image_exists"] = df["image_path"].apply(lambda p: Path(p).is_file())

missing = (~df["image_exists"]).sum()
if missing:
    print(f"Dropping {missing} dishes with no matching RGB image")
df = df[df["image_exists"]].drop(columns=["image_exists"]).reset_index(drop=True)
print(f"{len(df)} dishes remain with a usable image")

## 4. Train / test split

Splitting by `dish_id` so the same physical dish never appears in both
train and test (Nutrition5k has multiple camera angles/sessions per dish).

`# VERIFY:` whether the dataset repo ships an official split file
(`dish_ids/splits/*.txt` in past releases) -- using one would be preferable
to a random split for comparing against published baselines. Falling back
to a random 85/15 split with a fixed seed if none is found.

In [ ]:
official_split_dir = DATA_DIR / "dish_ids" / "splits"
train_ids, test_ids = None, None

if official_split_dir.is_dir():
    train_files = list(official_split_dir.glob("*train*ids*.txt"))
    test_files = list(official_split_dir.glob("*test*ids*.txt"))
    if train_files and test_files:
        train_ids = set(train_files[0].read_text().split())
        test_ids = set(test_files[0].read_text().split())
        print(f"Using official split: {len(train_ids)} train / {len(test_ids)} test")

if train_ids is None:
    print("No official split found -- using a random 85/15 split by dish_id.")
    ids = df["dish_id"].tolist()
    random.shuffle(ids)
    cutoff = int(len(ids) * 0.85)
    train_ids = set(ids[:cutoff])
    test_ids = set(ids[cutoff:])
    print(f"{len(train_ids)} train / {len(test_ids)} test")

train_df = df[df["dish_id"].isin(train_ids)].reset_index(drop=True)
test_df = df[df["dish_id"].isin(test_ids)].reset_index(drop=True)
print(f"train_df: {len(train_df)}  test_df: {len(test_df)}")

## 5. Build instruction-tuning JSONL (LLaVA-style conversations)

One example per dish: an image + a fixed instruction, with the assistant
turn containing the ground-truth calories/macros and a short ingredient
list, per the format in the spec doc.

In [ ]:
PROMPT = "Estimate the calories and macronutrients in this meal."

def format_target(row):
    ingredients = ", ".join(row["ingredients"][:6]) if row["ingredients"] else "unknown"
    return (
        f"About {row['calories']:.0f} kcal -- "
        f"{row['protein_g']:.0f}g protein, {row['carb_g']:.0f}g carbs, "
        f"{row['fat_g']:.0f}g fat. Likely ingredients: {ingredients}."
    )

def to_conversation(row):
    return {
        "image": row["image_path"],
        "dish_id": row["dish_id"],
        "calories": row["calories"],
        "protein_g": row["protein_g"],
        "carb_g": row["carb_g"],
        "fat_g": row["fat_g"],
        "conversations": [
            {"from": "user", "value": f"<image>\n{PROMPT}"},
            {"from": "assistant", "value": format_target(row)},
        ],
    }

def write_jsonl(frame, path):
    with open(path, "w") as f:
        for _, row in frame.iterrows():
            f.write(json.dumps(to_conversation(row)) + "\n")
    print(f"Wrote {len(frame)} examples to {path}")

Path("./data/processed").mkdir(parents=True, exist_ok=True)
write_jsonl(train_df, "./data/processed/train.jsonl")
write_jsonl(test_df, "./data/processed/test.jsonl")

## 6. Sanity check a sample

In [ ]:
with open("./data/processed/train.jsonl") as f:
    sample = json.loads(f.readline())

print(json.dumps(sample, indent=2))
display(Image.open(sample["image"]).resize((300, 300)))

## 7. Load the base model in 4-bit (QLoRA)

`Qwen2-VL-2B-Instruct` per the spec -- small enough to fine-tune on a
single free-tier GPU. Bump to the 7B variant later only if 2B accuracy is
insufficient after evaluation (Section 12).

In [ ]:
from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False

## 8. Apply LoRA

Fine-tuning only the language-model decoder layers (not the vision
encoder) -- standard practice for small VLM fine-tunes, and keeps memory
and overfitting risk down given the small (~5k example) dataset.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 9. Dataset & collate function

Builds the chat-formatted input via the processor's chat template, then
masks the prompt portion of `labels` with `-100` so the loss is only
computed on the assistant's response.

In [ ]:
from qwen_vl_utils import process_vision_info
from torch.utils.data import Dataset

class Nutrition5kDataset(Dataset):
    def __init__(self, jsonl_path):
        with open(jsonl_path) as f:
            self.examples = [json.loads(line) for line in f]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

def build_messages(example, include_answer):
    user_text = example["conversations"][0]["value"].replace("<image>\n", "")
    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": example["image"]},
            {"type": "text", "text": user_text},
        ]},
    ]
    if include_answer:
        messages.append({
            "role": "assistant",
            "content": [{"type": "text", "text": example["conversations"][1]["value"]}],
        })
    return messages

def collate_fn(batch):
    full_texts, prompt_texts, image_inputs_all = [], [], []

    for example in batch:
        full_messages = build_messages(example, include_answer=True)
        prompt_messages = build_messages(example, include_answer=False)

        full_text = processor.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)
        prompt_text = processor.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)

        image_inputs, _ = process_vision_info(full_messages)

        full_texts.append(full_text)
        prompt_texts.append(prompt_text)
        image_inputs_all.extend(image_inputs)

    inputs = processor(
        text=full_texts,
        images=image_inputs_all,
        padding=True,
        return_tensors="pt",
    )

    labels = inputs["input_ids"].clone()
    for i, prompt_text in enumerate(prompt_texts):
        prompt_len = len(processor.tokenizer(prompt_text, add_special_tokens=False)["input_ids"])
        labels[i, :prompt_len] = -100
    labels[inputs["attention_mask"] == 0] = -100
    inputs["labels"] = labels
    return inputs

train_dataset = Nutrition5kDataset("./data/processed/train.jsonl")
test_dataset = Nutrition5kDataset("./data/processed/test.jsonl")
print(f"train: {len(train_dataset)}  test: {len(test_dataset)}")

## 10. Training

Starting hyperparameters per the spec -- 3-5 epochs, small batch size with
gradient accumulation (the 2B model + images at 4-bit still takes real
memory per sample). Adjust `per_device_train_batch_size` /
`gradient_accumulation_steps` down further if you hit OOM on a T4.

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./checkpoints/qwen2vl-nutrition5k-lora",
    num_train_epochs=4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1.5e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
)

trainer.train()

## 11. Save the LoRA adapter

Saves locally; uncomment the `push_to_hub` line to also upload it (needs
the HF login from Section 1 with write access, and a repo name to push to).

In [ ]:
ADAPTER_DIR = "./checkpoints/qwen2vl-nutrition5k-lora-final"
model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)
print(f"Saved adapter to {ADAPTER_DIR}")

# model.push_to_hub("YOUR_HF_USERNAME/qwen2vl-nutrition5k-lora")
# processor.push_to_hub("YOUR_HF_USERNAME/qwen2vl-nutrition5k-lora")

## 12. Evaluate on the held-out test set

Runs inference on every test dish, extracts the predicted calorie number
from the generated text, and reports MAE / MAPE against ground truth --
the success criterion from the spec is beating the current Food-101 +
lookup-table baseline on this metric.

In [ ]:
model.eval()

def extract_calories(text):
    match = re.search(r"([\d,.]+)\s*kcal", text)
    if not match:
        return None
    return float(match.group(1).replace(",", ""))

def predict(example):
    messages = build_messages(example, include_answer=False)
    prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    inputs = processor(text=[prompt_text], images=image_inputs, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=128, do_sample=False)

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )[0]
    return generated

results = []
for example in test_dataset:
    generated_text = predict(example)
    predicted_kcal = extract_calories(generated_text)
    results.append({
        "dish_id": example["dish_id"],
        "true_kcal": example["calories"],
        "predicted_kcal": predicted_kcal,
        "raw_output": generated_text,
    })

results_df = pd.DataFrame(results)
valid = results_df.dropna(subset=["predicted_kcal"])
print(f"{len(valid)}/{len(results_df)} test examples produced a parseable calorie value")

mae = (valid["true_kcal"] - valid["predicted_kcal"]).abs().mean()
mape = ((valid["true_kcal"] - valid["predicted_kcal"]).abs() / valid["true_kcal"]).mean() * 100
print(f"MAE:  {mae:.1f} kcal")
print(f"MAPE: {mape:.1f}%")

results_df.to_csv("./data/processed/eval_results.csv", index=False)
results_df.head(10)

## 13. Next steps

- Compare `mae`/`mape` above against the current Food-101 + lookup-table
  baseline ([api/analyze-food.js](../api/analyze-food.js)) on the same test
  images, per the spec's evaluation section, before replacing the
  production path.
- If accuracy is insufficient: try the 7B base model, more epochs, or a
  higher LoRA rank before concluding the approach doesn't work.
- Hosting the trained adapter: see "Hosting & deployment" in
  [`docs/spec-vlm-calorie-estimation.md`](../docs/spec-vlm-calorie-estimation.md)
  -- start with a free Hugging Face Space on ZeroGPU, fall back to a paid
  autoscale-to-zero Inference Endpoint only if that proves unreliable for
  daily use.